# RAG 08: Chunking Strategies

This notebook compares chunking strategies before we create embeddings.

Chunking is not just a formatting step. It changes what the retriever can find later.

In [ ]:
from pathlib import Path
import json
import os
import re

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent


def repo_path(value: str) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return REPO_ROOT / path


PAGES_PATH = repo_path(os.getenv("OWASP_LLM_PAGES_PATH", "data/owasp_top10_llm_pages.jsonl"))
CHUNKS_PATH = repo_path(os.getenv("OWASP_LLM_CHUNKS_PATH", "data/owasp_top10_llm_chunks.jsonl"))

if not PAGES_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PAGES_PATH}. Run notebooks/07_rag_pdf_parsing_cleaning.ipynb first."
    )

In [ ]:
def load_documents(path: Path) -> list[Document]:
    documents = []
    with path.open(encoding="utf-8") as file:
        for line in file:
            row = json.loads(line)
            documents.append(Document(page_content=row["content"], metadata=row["metadata"]))
    return documents

page_documents = load_documents(PAGES_PATH)

print({"page_documents": len(page_documents)})

## Strategy 1: Page As Chunk

The simplest chunk is one PDF page. This preserves page citations, but one page can contain several different ideas.

In [ ]:
page_chunks = []

for index, document in enumerate(page_documents):
    metadata = dict(document.metadata)
    metadata.update({"chunk_id": f"page-{index:04d}", "chunk_strategy": "page"})
    page_chunks.append(Document(page_content=document.page_content, metadata=metadata))

print({"strategy": "page", "chunks": len(page_chunks)})
print(page_chunks[0].metadata)
print(page_chunks[0].page_content[:500])

## Strategy 2: Fixed-Size Character Chunks

Fixed-size chunks are easy to implement, but they ignore sentence and section boundaries.

In [ ]:
def fixed_size_chunks(documents: list[Document], chunk_size: int = 900) -> list[Document]:
    chunks = []

    for document in documents:
        text = document.page_content
        for start in range(0, len(text), chunk_size):
            content = text[start : start + chunk_size].strip()
            if not content:
                continue

            metadata = dict(document.metadata)
            metadata.update(
                {
                    "chunk_id": f"fixed-{len(chunks):04d}",
                    "chunk_strategy": "fixed_size",
                    "start_char": start,
                }
            )
            chunks.append(Document(page_content=content, metadata=metadata))

    return chunks

fixed_chunks = fixed_size_chunks(page_documents)

print({"strategy": "fixed_size", "chunks": len(fixed_chunks)})
print(fixed_chunks[3].metadata)
print(fixed_chunks[3].page_content[:500])

## Strategy 3: Recursive Chunks

Recursive splitting tries to preserve larger text boundaries before falling back to smaller ones. This is the default strategy for the final index.

In [ ]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    separators=["\n## ", "\n# ", "\n\n", "\n", ". ", " ", ""],
)

recursive_chunks = recursive_splitter.split_documents(page_documents)

for index, chunk in enumerate(recursive_chunks):
    chunk.metadata = dict(chunk.metadata)
    chunk.metadata.update(
        {
            "chunk_id": f"recursive-{index:04d}",
            "chunk_strategy": "recursive",
        }
    )

print({"strategy": "recursive", "chunks": len(recursive_chunks)})
print(recursive_chunks[3].metadata)
print(recursive_chunks[3].page_content[:500])

## Strategy 4: Heading-Aware Chunks

PDF-extracted headings are not always reliable. This simple strategy is useful for comparison, but recursive chunks remain the final indexing choice.

In [ ]:
heading_pattern = re.compile(r"^(LLM\d{2}|A\d{2}|[0-9]+\.|[A-Z][A-Z /-]{6,})")


def heading_aware_chunks(documents: list[Document]) -> list[Document]:
    chunks = []

    for document in documents:
        current_lines = []
        for line in document.page_content.splitlines():
            stripped = line.strip()
            if not stripped:
                continue

            starts_new_section = bool(heading_pattern.match(stripped)) and current_lines
            if starts_new_section:
                metadata = dict(document.metadata)
                metadata.update(
                    {
                        "chunk_id": f"heading-{len(chunks):04d}",
                        "chunk_strategy": "heading_aware",
                    }
                )
                chunks.append(Document(page_content="\n".join(current_lines), metadata=metadata))
                current_lines = []

            current_lines.append(stripped)

        if current_lines:
            metadata = dict(document.metadata)
            metadata.update(
                {
                    "chunk_id": f"heading-{len(chunks):04d}",
                    "chunk_strategy": "heading_aware",
                }
            )
            chunks.append(Document(page_content="\n".join(current_lines), metadata=metadata))

    return chunks

heading_chunks = heading_aware_chunks(page_documents)

print({"strategy": "heading_aware", "chunks": len(heading_chunks)})
print(heading_chunks[3].metadata)
print(heading_chunks[3].page_content[:500])

## Compare Chunk Boundaries

Before embeddings, inspect whether chunks look like useful pieces of knowledge.

In [ ]:
strategies = {
    "page": page_chunks,
    "fixed_size": fixed_chunks,
    "recursive": recursive_chunks,
    "heading_aware": heading_chunks,
}

for name, chunks in strategies.items():
    lengths = [len(chunk.page_content) for chunk in chunks]
    print(
        {
            "strategy": name,
            "chunks": len(chunks),
            "min_chars": min(lengths),
            "max_chars": max(lengths),
            "avg_chars": round(sum(lengths) / len(lengths)),
        }
    )

## Save Final Chunks

The final index uses recursive chunks because they give a good default balance: large enough for context, small enough for focused retrieval.

In [ ]:
CHUNKS_PATH.parent.mkdir(parents=True, exist_ok=True)

with CHUNKS_PATH.open("w", encoding="utf-8") as file:
    for chunk in recursive_chunks:
        file.write(
            json.dumps(
                {
                    "id": chunk.metadata["chunk_id"],
                    "content": chunk.page_content,
                    "metadata": chunk.metadata,
                },
                ensure_ascii=False,
            )
            + "\n"
        )

print({"chunks_path": str(CHUNKS_PATH), "chunks_saved": len(recursive_chunks)})